# WhisperColab жазу кітапшасы

Бұл ноутбук аудио немесе бейне файлдарын Whisperарқылы транскрипциялайды.

## Ерекшеліктер

- **Қолдану тегін:** Бұл жазу кітапшасы OpenAIтегін GPUжұмыс уақытымен бірге 2023 жылдың 17 қаңтарында ашық көзі болып табылатын OpenAIпайдаланады.
- **Ноутбук жағындағы файл өлшеміне бекітілген шектеу жоқ:** Үлкен аудио және бейне файлдарды өңдеуге болады.
- **Пакеттік өңдеу:** Бірнеше аудио және бейне файлдарды бір жұмыста өңдеуге болады.
- **Google Driveқолдау:** Файлдарды тікелей жүктеп салуға болады немесе Google Driveішінде сақталған аудио/бейне файлдарды өңдеуге болады.
- **Толық онлайн:** Жүктеп алу және өңдеу Google Colabмекенжайында орындалады. Жергілікті есептеуіш ресурстар пайдаланылмайды.
- **Бірнеше шығыс пішімдері:** Ноутбук түпнұсқа JSONтранскрипциясын және туынды TXT, CSV, Markdown, Excelжәне SRTсубтитр файлдарын шығарады.

## Қалай пайдалануға болады

1. **1 жылы. Параметрлер**, `INPUT_MODE`таңдаңыз және қажет болса үлгіні немесе шығыс опцияларын реттеңіз.
2. **Орындалу уақыты -> Барлығын іске қосу** таңдаңыз.
3. `INPUT_MODE = "upload"`болса, **2 дейін күтіңіз. Файлды таңдау / Google Driveорнату** басталады, содан кейін ұяшықтың шығыс аймағында **Файлдарды таңдау** түймесін басыңыз.
4. Өңдеу аяқталған кезде, `AUTO_DOWNLOAD_ZIP`қосылған болса, шығыс файлдары сығымдалады және жүктеледі.

## Ескертпелер

- Google Colabтегін GPUқолжетімділігі шектеулі. Егер GPUқолжетімді болмаса және `REQUIRE_CUDA = True`, жазу кітапшасы транскрипциядан бұрын тоқтайды.
- Өңдеу уақытты алады. large-v3-turboүлгісін пайдалану кезінде дөрекі нұсқаулық ретінде 60 минуттық аудио немесе бейне файлды транскрипциялауға шамамен 5-10 минут кетуі мүмкін. Егер файлдарды тікелей жүктеп салсаңыз, жүктеп салу уақыты да қажет.
- Өңдеу кезінде осы жазу кітапшасын ашық ұстаңыз. Транскрипция кезінде жазу кітапшасын жапсаңыз, өңдеу тоқтайды.


In [ ]:
# @title 1. Параметрлер
# ============================================================
# 1. Параметрлер
# ============================================================

from pathlib import Path

# @markdown ## Енгізу
# @markdown - `INPUT_MODE`: Жергілікті файлдар үшін `upload`немесе Google Driveішінде сақталған файлдар үшін `google_drive`пайдаланыңыз.
INPUT_MODE = "upload"  # @param ["upload", "google_drive"]

# @markdown ### Google Driveкірісі
# @markdown Бұл параметрлер тек `INPUT_MODE == "google_drive"`кезінде пайдаланылады.
# @markdown - `GOOGLE_DRIVE_ROOT`: Бұл жиынды әдетте `/content/drive/MyDrive`етіп сақтаңыз.
# @markdown - `GOOGLE_DRIVE_SUBPATH`: MyDrive астындағы файл немесе қалта жолы. Жоғарғы деңгейлі MyDrive қалтасын пайдалану үшін бос қалдырыңыз.
# @markdown - `RECURSIVE`: Drive енгізу жолы қалта болса, ішкі қалталарды да іздеңіз.
# @markdown - `FILENAME_CONTAINS`: Қосымша файл атауы ішкі жол сүзгісі.
GOOGLE_DRIVE_ROOT = "/content/drive/MyDrive"  # @param {type:"string"}
GOOGLE_DRIVE_SUBPATH = ""  # @param {type:"string"}
RECURSIVE = False  # @param {type:"boolean"}
FILENAME_CONTAINS = ""  # @param {type:"string"}

_drive_root = str(Path(GOOGLE_DRIVE_ROOT.strip().rstrip("/")))
_drive_subpath = GOOGLE_DRIVE_SUBPATH.strip().strip("/")
DRIVE_INPUT_PATH = str(Path(_drive_root) / _drive_subpath) if _drive_subpath else _drive_root

# @markdown ## Whisperүлгісі
# @markdown - `MODEL_ID`: Hugging Faceүлгі идентификаторы. Турбо жылдамырақ; large-v3кейбір файлдар үшін тұрақтырақ болуы мүмкін.
# @markdown - `SOURCE_LANGUAGE`: Белгісіз кезде `auto`пайдаланыңыз немесе белгілі бастапқы тілді таңдаңыз.
# @markdown - `TRANSLATE_TO_ENGLISH`: Ағылшын тілін шығару үшін Whisperаудару режимін пайдаланыңыз.
MODEL_ID = "openai/whisper-large-v3-turbo"  # @param ["openai/whisper-large-v3-turbo", "openai/whisper-large-v3", "openai/whisper-medium", "openai/whisper-small"]
SOURCE_LANGUAGE = "kazakh"  # @param ["auto", "english", "japanese", "chinese", "german", "spanish", "russian", "korean", "french", "portuguese", "turkish", "polish", "catalan", "dutch", "arabic", "swedish", "italian", "indonesian", "hindi", "finnish", "vietnamese", "hebrew", "ukrainian", "greek", "malay", "czech", "romanian", "danish", "hungarian", "tamil", "norwegian", "thai", "urdu", "croatian", "bulgarian", "lithuanian", "latin", "maori", "malayalam", "welsh", "slovak", "telugu", "persian", "latvian", "bengali", "serbian", "azerbaijani", "slovenian", "kannada", "estonian", "macedonian", "breton", "basque", "icelandic", "armenian", "nepali", "mongolian", "bosnian", "kazakh", "albanian", "swahili", "galician", "marathi", "punjabi", "sinhala", "khmer", "shona", "yoruba", "somali", "afrikaans", "occitan", "georgian", "belarusian", "tajik", "sindhi", "gujarati", "amharic", "yiddish", "lao", "uzbek", "faroese", "haitian creole", "pashto", "turkmen", "nynorsk", "maltese", "sanskrit", "luxembourgish", "myanmar", "tibetan", "tagalog", "malagasy", "assamese", "tatar", "hawaiian", "lingala", "hausa", "bashkir", "javanese", "sundanese", "cantonese"]
TRANSLATE_TO_ENGLISH = False  # @param {type:"boolean"}

# @markdown ## Шығару
# @markdown - `SAVE_OUTPUTS_TO_GOOGLE_DRIVE`: Drive кірісі пайдаланылған кезде шығыстарды Google Driveастында сақтаңыз.
# @markdown - `AUTO_DOWNLOAD_ZIP`: Соңында бір ZIPмұрағатын жасаңыз және жүктеңіз.
# - Шығару пішімдері төмендегі логикалық мәндермен басқарылады.
SAVE_OUTPUTS_TO_GOOGLE_DRIVE = True  # @param {type:"boolean"}
AUTO_DOWNLOAD_ZIP = True  # @param {type:"boolean"}
SAVE_TXT = True
SAVE_MD = True
SAVE_JSON = True
SAVE_CSV = True
SAVE_XLSX = True
SAVE_SRT = True
INCLUDE_TIMESTAMPS_IN_TXT = True

# ## Жетілдірілген
# Бұл әдепкі мәндер Colab үшін консервативті болып табылады. Әдетте қарапайым пайдаланушыларға оларды өңдеу қажет емес.
REQUIRE_CUDA = True
PIPELINE_CHUNK_LENGTH_SECONDS = 30
PIPELINE_BATCH_SIZE = 8
MAX_SEGMENT_SECONDS = 0
MODEL_ATTN_IMPLEMENTATION = ""
UPLOAD_DIR = "/content/whisper_uploads"
WORK_DIR = "/content/whisper_work"
OUTPUT_DIR = "/content/whisper_outputs"
ZIP_FILE_NAME = "whisper_outputs.zip"
SUPPORTED_EXTS = {
    ".aac", ".avi", ".flac", ".m4a", ".m4v", ".mkv", ".mov", ".mp3",
    ".mp4", ".ogg", ".opus", ".wav", ".webm", ".wma"
}

print("Параметрлер жүктелді.")
print(f"INPUT_MODE={INPUT_MODE}")
print(f"DRIVE_INPUT_PATH={DRIVE_INPUT_PATH}")
print(f"MODEL_ID={MODEL_ID}")
print(f"SOURCE_LANGUAGE={SOURCE_LANGUAGE}")
print(f"TRANSLATE_TO_ENGLISH={TRANSLATE_TO_ENGLISH}")


In [ ]:
# @title 2. Файлды таңдау / Google Driveорнату
# ============================================================
# 2. Файлды таңдау / Google Driveорнату
# ============================================================

from pathlib import Path
import shutil

Path(UPLOAD_DIR).mkdir(parents=True, exist_ok=True)
Path(WORK_DIR).mkdir(parents=True, exist_ok=True)
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)


def is_supported_media(path: Path) -> bool:
    return path.is_file() and path.suffix.lower() in SUPPORTED_EXTS


def collect_drive_files(path_str: str) -> list[str]:
    root = Path(path_str)
    if root.is_file():
        if not is_supported_media(root):
            raise ValueError(f"Unsupported file type: {root}")
        return [str(root)]
    if not root.exists():
        raise FileNotFoundError(f"Drive path does not exist: {root}")
    iterator = root.rglob("*") if RECURSIVE else root.glob("*")
    files = []
    for path in iterator:
        if is_supported_media(path):
            if FILENAME_CONTAINS and FILENAME_CONTAINS not in path.name:
                continue
            files.append(str(path))
    return sorted(files)


input_files = []

if INPUT_MODE.lower() == "upload":
    from google.colab import files
    print("«Файлдарды таңдау» түймесін басып, бір немесе бірнеше аудио/бейне файлдарын жүктеп салыңыз.")
    uploaded = files.upload()
    for name in uploaded.keys():
        src = Path(name)
        dst = Path(UPLOAD_DIR) / src.name
        if src.exists():
            shutil.move(str(src), str(dst))
        if is_supported_media(dst):
            input_files.append(str(dst))
        else:
            print(f"Skipped unsupported file: {dst.name}")
elif INPUT_MODE.lower() == "google_drive":
    from google.colab import drive
    drive.mount("/content/drive")
    input_files = collect_drive_files(DRIVE_INPUT_PATH)
else:
    raise ValueError(f"Unsupported INPUT_MODE: {INPUT_MODE}")

if not input_files:
    raise ValueError("Қолдау көрсетілетін аудио/бейне файлдары таңдалмады.")

print("Таңдалған файлдар:")
for file_path in input_files:
    print("-", file_path)


In [ ]:
# @title 3. GPUтексеру және тәуелділікті орнату
# ============================================================
# 3. GPUтексеру және тәуелділікті орнату
# ============================================================

import shutil
import subprocess
import sys


def run_command(command: list[str]) -> None:
    print("$", " ".join(command))
    subprocess.run(command, check=True)


if not shutil.which("ffmpeg"):
    run_command(["apt-get", "update", "-qq"])
    run_command(["apt-get", "install", "-y", "ffmpeg"])

flag_path = Path("/content/whisper_dependencies_installed.flag")
if not flag_path.exists():
    run_command([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])
    run_command([
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "transformers",
        "accelerate",
        "datasets[audio]",
        "pandas",
        "openpyxl",
    ])
    flag_path.write_text("installed", encoding="utf-8")

import torch

if REQUIRE_CUDA and not torch.cuda.is_available():
    raise RuntimeError("This notebook requires a CUDA GPU. In Colab, select Runtime > Change runtime type > Hardware accelerator > GPU, then restart and rerun.")

print("Тәуелділікті орнату аяқталды.")
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# @title 4. Көмекші функциялары
# ============================================================
# 4. Көмекші функциялары
# ============================================================

from pathlib import Path
import csv
import json
import subprocess
import zipfile


def safe_stem(path: Path) -> str:
    return path.name.replace("/", "_").replace("\\\\", "_")


def run_ffmpeg(command: list[str]) -> None:
    completed = subprocess.run(command, capture_output=True, text=True, check=False)
    if completed.returncode != 0:
        detail = (completed.stderr or completed.stdout or "ffmpeg failed").strip()
        raise RuntimeError(detail)


def extract_audio_for_whisper(src_path: str) -> Path:
    src = Path(src_path)
    out_dir = Path(WORK_DIR) / "audio"
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{src.stem}.whisper.wav"
    command = [
        "ffmpeg", "-y", "-hide_banner", "-loglevel", "error", "-nostdin",
        "-i", str(src), "-map", "0:a:0", "-vn", "-ac", "1", "-ar", "16000",
        "-c:a", "pcm_s16le", str(out_path),
    ]
    run_ffmpeg(command)
    if not out_path.exists() or out_path.stat().st_size < 1000:
        raise RuntimeError(f"Extracted audio is missing or too small: {out_path}")
    return out_path


def split_audio(audio_path: Path) -> list[Path]:
    max_seconds = int(MAX_SEGMENT_SECONDS or 0)
    if max_seconds <= 0:
        return [audio_path]
    segment_dir = Path(WORK_DIR) / "segments" / audio_path.stem
    segment_dir.mkdir(parents=True, exist_ok=True)
    for stale in segment_dir.glob("part*.wav"):
        stale.unlink()
    pattern = segment_dir / "part%03d.wav"
    command = [
        "ffmpeg", "-y", "-hide_banner", "-loglevel", "error", "-nostdin",
        "-i", str(audio_path), "-f", "segment", "-segment_time", str(max_seconds),
        "-reset_timestamps", "1", "-c", "copy", str(pattern),
    ]
    run_ffmpeg(command)
    segments = sorted(segment_dir.glob("part*.wav"))
    if not segments:
        raise RuntimeError(f"No audio segments were created in {segment_dir}")
    return segments


def timestamp_seconds(value):
    if value is None:
        return None
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def format_timestamp(seconds) -> str:
    seconds = timestamp_seconds(seconds) or 0
    total_ms = int(round(seconds * 1000))
    hours, rem = divmod(total_ms, 3600000)
    minutes, rem = divmod(rem, 60000)
    secs, millis = divmod(rem, 1000)
    return f"{hours:02d}:{minutes:02d}:{secs:02d}.{millis:03d}"


def format_srt_timestamp(seconds) -> str:
    return format_timestamp(seconds).replace(".", ",")


def offset_transcription(transcription: dict, offset: int) -> dict:
    if offset <= 0:
        return transcription
    shifted = dict(transcription)
    chunks = []
    for chunk in transcription.get("chunks") or []:
        next_chunk = dict(chunk)
        timestamp = next_chunk.get("timestamp")
        if isinstance(timestamp, (list, tuple)):
            next_chunk["timestamp"] = tuple(
                None if item is None else timestamp_seconds(item) + offset for item in timestamp
            )
        chunks.append(next_chunk)
    shifted["chunks"] = chunks
    return shifted


def merge_transcriptions(parts: list[dict]) -> dict:
    if len(parts) == 1:
        return parts[0]
    text_parts = []
    chunks = []
    for part in parts:
        text = str(part.get("text", "")).strip()
        if text:
            text_parts.append(text)
        chunks.extend(part.get("chunks") or [])
    return {"text": " ".join(text_parts), "chunks": chunks}


def language_for_generate_kwargs():
    language = str(SOURCE_LANGUAGE or "auto").strip().lower()
    if language in {"", "auto", "none"}:
        return None
    return language


def build_generate_kwargs() -> dict:
    kwargs = {"task": "translate" if TRANSLATE_TO_ENGLISH else "transcribe"}
    language = language_for_generate_kwargs()
    if language is not None:
        kwargs["language"] = language
    return kwargs


def transcript_rows(transcription: dict) -> list[dict]:
    chunks = transcription.get("chunks") or []
    if not chunks:
        return [{"start": None, "end": None, "text": transcription.get("text", "")}]
    rows = []
    for chunk in chunks:
        timestamp = chunk.get("timestamp") or (None, None)
        start = timestamp[0] if len(timestamp) >= 1 else None
        end = timestamp[1] if len(timestamp) >= 2 else None
        rows.append({"start": start, "end": end, "text": str(chunk.get("text", "")).strip()})
    return rows


def write_txt(path: Path, transcription: dict) -> None:
    rows = transcript_rows(transcription)
    lines = []
    for row in rows:
        text = str(row["text"]).strip()
        if INCLUDE_TIMESTAMPS_IN_TXT and row["start"] is not None:
            lines.append(f"[{format_timestamp(row['start'])}] {text}")
        else:
            lines.append(text)
    path.write_text("\n".join(lines).strip() + "\n", encoding="utf-8")


def write_csv(path: Path, transcription: dict) -> None:
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=["start", "end", "text"])
        writer.writeheader()
        writer.writerows(transcript_rows(transcription))


def write_md(path: Path, source_path: str, transcription: dict) -> None:
    lines = [f"# {Path(source_path).name}", ""]
    for row in transcript_rows(transcription):
        text = str(row["text"]).strip()
        if not text:
            continue
        if row["start"] is not None:
            lines.append(f"- **{format_timestamp(row['start'])}** {text}")
        else:
            lines.append(text)
    path.write_text("\n".join(lines).rstrip() + "\n", encoding="utf-8")


def write_xlsx(path: Path, transcription: dict) -> None:
    import pandas as pd
    pd.DataFrame(transcript_rows(transcription)).to_excel(path, index=False)


def write_srt(path: Path, transcription: dict) -> None:
    lines = []
    for index, row in enumerate(transcript_rows(transcription), start=1):
        if row["start"] is None:
            continue
        end = row["end"] if row["end"] is not None else row["start"]
        lines.extend([
            str(index),
            f"{format_srt_timestamp(row['start'])} --> {format_srt_timestamp(end)}",
            str(row["text"]).strip(),
            "",
        ])
    path.write_text("\n".join(lines), encoding="utf-8")


def output_dir_for_source(source_path: str) -> Path:
    source = Path(source_path)
    if INPUT_MODE.lower() == "google_drive" and SAVE_OUTPUTS_TO_GOOGLE_DRIVE:
        output_dir = Path(GOOGLE_DRIVE_ROOT) / "Whisper_outputs"
    else:
        output_dir = Path(OUTPUT_DIR)
    output_dir.mkdir(parents=True, exist_ok=True)
    return output_dir


def save_outputs(source_path: str, transcription: dict) -> list[Path]:
    source = Path(source_path)
    output_dir = output_dir_for_source(source_path)
    base = safe_stem(source)
    saved = []
    if SAVE_TXT:
        path = output_dir / f"{base}.txt"
        write_txt(path, transcription)
        saved.append(path)
    if SAVE_MD:
        path = output_dir / f"{base}.md"
        write_md(path, source_path, transcription)
        saved.append(path)
    if SAVE_JSON:
        path = output_dir / f"{base}.json"
        path.write_text(json.dumps(transcription, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
        saved.append(path)
    if SAVE_CSV:
        path = output_dir / f"{base}.csv"
        write_csv(path, transcription)
        saved.append(path)
    if SAVE_XLSX:
        path = output_dir / f"{base}.xlsx"
        write_xlsx(path, transcription)
        saved.append(path)
    if SAVE_SRT:
        path = output_dir / f"{base}.srt"
        write_srt(path, transcription)
        saved.append(path)
    for path in saved:
        print("Saved:", path)
    return saved


def create_zip(output_paths: list[Path], zip_path: Path) -> Path:
    zip_path.parent.mkdir(parents=True, exist_ok=True)
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in output_paths:
            archive.write(path, arcname=path.name)
    return zip_path

print("Helper functions loaded.")


In [ ]:
# @title 5. Whisperжүктеп, транскрипцияны іске қосыңыз
# ============================================================
# 5. Whisperжүктеп, транскрипцияны іске қосыңыз
# ============================================================

import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model_kwargs = {
    "low_cpu_mem_usage": True,
    "use_safetensors": True,
}
if str(MODEL_ATTN_IMPLEMENTATION).strip():
    model_kwargs["attn_implementation"] = str(MODEL_ATTN_IMPLEMENTATION).strip()

try:
    model = AutoModelForSpeechSeq2Seq.from_pretrained(MODEL_ID, dtype=torch_dtype, **model_kwargs)
except TypeError:
    model = AutoModelForSpeechSeq2Seq.from_pretrained(MODEL_ID, torch_dtype=torch_dtype, **model_kwargs)
model.to(device)
processor = AutoProcessor.from_pretrained(MODEL_ID)

pipeline_kwargs = {
    "task": "automatic-speech-recognition",
    "model": model,
    "tokenizer": processor.tokenizer,
    "feature_extractor": processor.feature_extractor,
    "return_timestamps": True,
    "device": device,
}
if int(PIPELINE_CHUNK_LENGTH_SECONDS or 0) > 0:
    pipeline_kwargs["chunk_length_s"] = int(PIPELINE_CHUNK_LENGTH_SECONDS)
if int(PIPELINE_BATCH_SIZE or 0) > 0:
    pipeline_kwargs["batch_size"] = int(PIPELINE_BATCH_SIZE)

try:
    asr_pipe = pipeline(**pipeline_kwargs, dtype=torch_dtype)
except TypeError:
    asr_pipe = pipeline(**pipeline_kwargs, torch_dtype=torch_dtype)

print("Whisperүлгісі жүктелді.")

generate_kwargs = build_generate_kwargs()
all_saved_files = []
results = []
total_files = len(input_files)

for file_index, source_path in enumerate(input_files, start=1):
    print(f"[{file_index}/{total_files}] Extracting audio: {source_path}")
    audio_path = extract_audio_for_whisper(source_path)
    segment_paths = split_audio(audio_path)
    parts = []
    for segment_index, segment_path in enumerate(segment_paths, start=1):
        print(f"[{file_index}/{total_files}] Transcribing segment {segment_index}/{len(segment_paths)}: {segment_path}")
        transcription = asr_pipe(str(segment_path), return_timestamps=True, generate_kwargs=generate_kwargs)
        offset = (segment_index - 1) * int(MAX_SEGMENT_SECONDS or 0)
        parts.append(offset_transcription(transcription, offset))
    merged = merge_transcriptions(parts)
    saved = save_outputs(source_path, merged)
    all_saved_files.extend(saved)
    results.append({"source_path": source_path, "saved_files": [str(path) for path in saved]})

print("Транскрипция аяқталды.")
for result in results:
    print("-", result["source_path"])


In [ ]:
# @title 6. Zip және жүктеп алу шығыстары
# ============================================================
# 6. Zip және жүктеп алу шығыстары
# ============================================================

if not all_saved_files:
    print("No output files to zip.")
else:
    zip_path = create_zip(all_saved_files, Path(OUTPUT_DIR) / ZIP_FILE_NAME)
    print("ZIP:", zip_path)
    if AUTO_DOWNLOAD_ZIP:
        from google.colab import files
        files.download(str(zip_path))
